<div style="width:100%; text-align:center; padding:10px 0;">
<img src="project_header.png" style="width:100%; max-width:100vw; height:auto; display:block; margin:0 auto;">
</div>

# EO Africa SWAM project 
## Batch-process ACOLITE RAdCor Atmospheric correction
In the SWAM project we use the remote sensing reflectance data produced by the ACOLITE RAdCor atmospheric correction as input for our Chl-a algorith. This note book creates the output remote sensing reflectance products. 

#### This notebook does the following:
* reads in the Sentinel-2 L1C filenames from the textfile
* Runs the ACOLITE atmospheric correction processor, which performs all the following steps:
<br> 1. read the file 
<br> 2. resample the radiance to user-defined spatial resolution of 20m 
<br> 3. subset to the user-defined region of interest
<br> 4. apply the DSF residual glint correction
<br> 5. apply the RAdCor atmospheric correction 
<br> 6. writes all output to a folder containing a netcdf file (*_L2W.nc) with Rrs bands

#### Requirements 
* Make sure that you have already created a "**LAKE**_filenames.txt" with the *Chl_Step1_Create_list_of_filenames.ipynb* notebook.
* Make sure that you have either cloned or copied an ACOLITE installation (we used **acolite20250402**) into your working directory. Follow the installation instructions from the acolite [github repository](https://github.com/acolite) 

#### Settings to adjust manually:
* in code cell 1 define the path to acolite installation
* in code cell 2 define the lake/dam name  (options: ZV, RV, TW, MV, VV, CW), default is TW.
* Input and output paths are selected and created automatically
* in code cell 4 copy and paste your selected lake/dam "limits" from the regions of interest supplied (default is for TW)
* in code cell 4 adjust your spatial resolution "s2_target_res" as needed (default 20m)

##### Authors:
**Marie Smith**, CSIR, South Africa

In [ ]:
import os
import sys
import glob
# change the path below to where the Acolite code is saved
sys.path.append("/SWAM/acolite20250402")

import acolite as ac

In [ ]:
## supply the name of the lake (options: ZV, RV, TW, MV, VV, CW; default TW)
lake_name = "TW"    

## create the folder where the outputs will be saved:
if not os.path.exists(lake_name+'_acolite'):
    os.makedirs(lake_name+'_acolite')
    print(f"Created folder: {lake_name}_acolite")
else:
    print(f"Folder {lake_name}_acolite already exists.") [web:1][web:5]
    
output_path = "/"+lake_name+"_acolite/"         

Read in the textfile containing the filenames that you want to process:

In [ ]:
with open(lake_name+'_filenames.txt', 'r') as file:
    filenames = file.read().splitlines()
num = len(filenames)
print(str(num) + ' filtered files total')  

Below is the main processing step where you read the input file location, apply the acolite atmospheric correction and write the output files. Each file can take a few minutes to complete, so be patient! 

Important to note that the RAdCor processor sometimes crops the image to a smaller region than you originally defined, so it is useful to start off with a slightly larger region than you were initially planning (by approximately 0.05 degrees in each direction)

In [ ]:
for i in range(0, num):
    i_file = filenames[i]
    in_file = os.path.dirname(i_file)
    img_info = os.path.basename(in_file)
    info_txt = img_info.split("_")
    out_file = out_folder+info_txt[0]+"_"+info_txt[2]+"_"+info_txt[5]+ "_"+lake_name+"_Acolite"
    
    print(">>>>>>>>>>>>>>>>>>>>>>>>>>>>> start to process " + str(i+1) + "/" + str(num) + ": " + img_info + " >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>")      
    my_settings = {"inputfile": in_file,
                  "output": out_file,
                   "limit":[-34.1482, 19.0489, -33.9152, 19.3681],  # region of interest in format: Lat(S), Lon(W), Lat(N), Lon(E)
                   # Theewaterskloof is [-34.1482, 19.0489, -33.9152, 19.3681]
                   # Zeekoevlei is [-34.15, 18.4, -33.97, 18.6], 
                   # Rietvlei is [-33.8995, 18.4275, -33.7820, 18.5567],
                   # Misverstand is [-33.26, 18.64, -32.95, 19] 
                   # Voelvlei is [-33.45, 18.95, -33.25, 19.15]
                   # Clanwilliam is [-32.35, 18.8, -32.1, 19] 
                   "l2w_parameters":"Rrs_*",      
                   "s2_target_res": 20,  # we set our spatial resolution to 20m
                   "l2w_mask_wave":1600,
                   "l2w_mask_threshold": 0.065,
                   "dsf_residual_glint_correction":True,  
                   "atmospheric_correction":True,
                   "atmospheric_correction_method":"radcor",
                   ## options for smaller output file:
                   "netcdf_compression":True,
                   "netcdf_compression_level":9,
                   "radcor_crop_center":False,
                   "rgb_rhot": False,
                   "rgb_rhotc": False,
                   "rgb_rhos": False,
                   "rgb_rhosu": False,
                   "rgb_rhorc": False,
                   "rgb_rhow": False,
                   "l1r_delete_netcdf": True,  # we don't have to keep the L1R file
                   "l2r_delete_netcdf": True   # we don't have to keep the L2R file
                  }

    ac.acolite.acolite_run(settings=my_settings)
    print("***************************** finished processing: " + img_info + " ***********************************") 

print("======================================================= well done! all finished ============================================================================")    
    